In [ ]:
txt_file_path = '/hkfs/home/project/hk-project-p00201316/st_st176945/testin/test.txt'
json_file_path = '/hkfs/home/project/hk-project-p00201316/st_st176945/new_para_cache/para_cache.json'

In [ ]:
import re
import json
from tqdm import tqdm

# Define a function to extract the first two entity IDs
def extract_first_two_entity_ids(line):
    pattern = r"\[entity\d, (Q\d+)\]"
    matches = re.findall(pattern, line)
    return matches[:2]  # Only the first two entity IDs are returned

# Read the file and extract the first two entity IDs in each line
def extract_ids_from_file(file_path):
    lines_with_ids = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in tqdm(file, desc="Extracting IDs"):
            ids = extract_first_two_entity_ids(line)
            if ids:
                lines_with_ids.append((line.strip(), ids))
    return lines_with_ids

# Define a function to read a JSON file and parse it into a dictionary
def read_json_as_dict(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data_dict = json.load(file)
    return data_dict

# Define a function to find the corresponding text based on an entity ID
def get_texts_for_entity_ids(entity_ids, data_dict):
    texts = []
    for entity_id in entity_ids:
        text = data_dict.get(entity_id, "Text not found")
        # Remove the entity information section
        cleaned_text = re.sub(r'^\[.*?\]:\s*', '', text)
        texts.append(cleaned_text)
    return texts

# Main function, integrating all steps
def main(txt_file_path, json_file_path, output_file_path):
    # Extracting entity IDs
    lines_with_ids = extract_ids_from_file(txt_file_path)
    
    # Reading JSON files and parsing them into dictionaries
    data_dict = read_json_as_dict(json_file_path)
    
    # Generate new content containing the original text and the corresponding entity text
    new_lines = []
    for line, ids in tqdm(lines_with_ids, desc="Processing lines"):
        texts = get_texts_for_entity_ids(ids, data_dict)
        new_line = line + " <|paragraphs|> <|paragraph|> " + " <|endofparagraph|> <|paragraph|> ".join(texts) + " <|endofparagraph|> <|endofparagraphs|>"
        new_lines.append(new_line)
    
    # Write new content to the output file
    with open(output_file_path, 'w', encoding='utf-8') as file:
        for new_line in tqdm(new_lines, desc="Writing output"):
            file.write(new_line + "\n")

txt_file_path = '/hkfs/home/project/hk-project-p00201316/st_st176945/testin/0.txt'
json_file_path = '/hkfs/home/project/hk-project-p00201316/st_st176945/new_para_cache/para_cache.json'
output_file_path = '/hkfs/home/project/hk-project-p00201316/st_st176945/testout/0.txt'

main(txt_file_path, json_file_path, output_file_path)


In [ ]:
#Batch processing
import os
import re
import json
from tqdm import tqdm

def extract_first_two_entity_ids(line):
    pattern = r"\[entity\d, (Q\d+)\]"
    matches = re.findall(pattern, line)
    return matches[:2]  

def extract_ids_from_file(file_path):
    lines_with_ids = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in tqdm(file, desc=f"Extracting IDs from {os.path.basename(file_path)}"):
            ids = extract_first_two_entity_ids(line)
            if ids:
                lines_with_ids.append((line.strip(), ids))
    return lines_with_ids

def read_json_as_dict(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data_dict = json.load(file)
    return data_dict

def get_texts_for_entity_ids(entity_ids, data_dict):
    texts = []
    for entity_id in entity_ids:
        text = data_dict.get(entity_id, "Text not found")
        cleaned_text = re.sub(r'^\[.*?\]:\s*', '', text)
        texts.append(cleaned_text)
    return texts

def process_file(txt_file_path, data_dict, output_file_path):

    lines_with_ids = extract_ids_from_file(txt_file_path)
    
    new_lines = []
    for line, ids in tqdm(lines_with_ids, desc="Processing lines"):
        texts = get_texts_for_entity_ids(ids, data_dict)
        new_line = line + " <|paragraphs|> <|paragraph|> " + " <|endofparagraph|> <|paragraph|> ".join(texts) + " <|endofparagraph|> <|endofparagraphs|>"
        new_lines.append(new_line)
    
    with open(output_file_path, 'w', encoding='utf-8') as file:
        for new_line in tqdm(new_lines, desc="Writing output"):
            file.write(new_line + "\n")

def batch_process_files(txt_directory, json_file_path, output_directory):

    data_dict = read_json_as_dict(json_file_path)
    
    os.makedirs(output_directory, exist_ok=True)
    
    for file_name in os.listdir(txt_directory):
        if file_name.endswith('.txt'):
            txt_file_path = os.path.join(txt_directory, file_name)
            output_file_path = os.path.join(output_directory, file_name)
            process_file(txt_file_path, data_dict, output_file_path)

txt_directory = '/hkfs/home/project/hk-project-p00201316/st_st176945/text2kg_and_hops'
json_file_path = '/hkfs/home/project/hk-project-p00201316/st_st176945/new_para_cache/para_cache.json'
output_directory = '/hkfs/home/project/hk-project-p00201316/st_st176945/Final_Merged_Text2KG'

batch_process_files(txt_directory, json_file_path, output_directory)
